# Mechanism Tutorial

This notebook shows the current multi-compartment mechanism API around `Ion` and `Channel`.

It focuses on three cases:

1. default fallback ions and automatic channel binding
2. named ion instances and explicit channel binding when one family is ambiguous
3. a mixed-ion channel (`K + Ca`) whose current belongs to potassium while calcium only modulates gating


In [1]:
import os

os.environ.setdefault("JAX_PLATFORMS", "cpu")

import brainunit as u
import braincell
from braincell import Branch, Morphology, Cell
from braincell.filter import AllRegion, BranchSlice, at
from braincell.mech import CableProperty, Channel, CurrentProbe, Ion


def build_demo_morphology():
    soma = Branch.from_lengths(
        lengths=[20.0] * u.um,
        radii=[8.0, 8.0] * u.um,
        type="soma",
    )
    dend = Branch.from_lengths(
        lengths=[60.0] * u.um,
        radii=[2.0, 1.2] * u.um,
        type="basal_dendrite",
    )
    axon = Branch.from_lengths(
        lengths=[80.0] * u.um,
        radii=[1.0, 0.6] * u.um,
        type="axon",
    )
    morpho = Morphology.from_root(soma, name="soma")
    morpho.attach(parent="soma", child_branch=dend, child_name="dend", parent_x=1.0)
    morpho.attach(parent="soma", child_branch=axon, child_name="axon", parent_x=0.0)
    return morpho


ERROR:2026-04-20 18:07:57,657:jax._src.xla_bridge:444: Jax plugin configuration error: Exception when calling jax_plugins.xla_cuda12.initialize()
Traceback (most recent call last):
  File "/home/swl/anaconda3/envs/braincell/lib/python3.10/site-packages/jax/_src/xla_bridge.py", line 442, in discover_pjrt_plugins
    plugin_module.initialize()
  File "/home/swl/anaconda3/envs/braincell/lib/python3.10/site-packages/jax_plugins/xla_cuda12/__init__.py", line 324, in initialize
    _check_cuda_versions(raise_on_first_error=True)
  File "/home/swl/anaconda3/envs/braincell/lib/python3.10/site-packages/jax_plugins/xla_cuda12/__init__.py", line 257, in _check_cuda_versions
    cublas_version = _version_check("cuBLAS", cuda_versions.cublas_get_version,
  File "/home/swl/anaconda3/envs/braincell/lib/python3.10/site-packages/jax_plugins/xla_cuda12/__init__.py", line 217, in _version_check
    raise RuntimeError(msg)
RuntimeError: Outdated cuBLAS installation found.
Version JAX was built against: 12

In [2]:
morpho = build_demo_morphology()
print(morpho.topo())

cell = Cell(morpho)
cell.paint(
    AllRegion(),
    CableProperty(
        resting_potential=-65.0 * u.mV,
        membrane_capacitance=1.0 * (u.uF / u.cm**2),
        axial_resistivity=100.0 * (u.ohm * u.cm),
    ),
)
print(cell)


soma
├── dend
└── axon
-----------------------------------
root           | soma
n_branches     | 3
n_cv           | 3
n_paint_rules  | 1
n_place_rules  | 0
-----------------------------------



## 1. Default ions and automatic channel binding

A fresh `Cell` still has fallback ion containers for `na`, `k`, and `ca`.
If one ion family has only one candidate, an ion-bound channel can bind automatically without any selector.

In [3]:
cell_auto = Cell(build_demo_morphology())
cell_auto.paint(
    BranchSlice(branch_index=0, prox=0.0, dist=1.0),
    Channel("INa_HH1952", g_max=12.0 * (u.mS / u.cm**2)),
)

layout = next(layout for layout in cell_auto.layouts if layout.kind == "channel:INa_HH1952")
node = cell_auto.get_runtime_node(layout.id)
na = cell_auto.get_ion("na")

print(type(na).__name__)
print(type(node).__name__)
print("bound to fallback na:", na.channels["INa_HH1952"] is node)


SodiumFixed
INa_HH1952
bound to fallback na: True


## 2. Named ion instances and explicit channel binding

When one family has multiple ion instances, family-level lookup becomes ambiguous.
Then a single-ion channel should declare `ion_name="..."`.

In [4]:
cell_named = Cell(build_demo_morphology())
cell_named.paint(
    BranchSlice(branch_index=0, prox=0.0, dist=1.0),
    Ion("SodiumFixed", name="na_soma", E=55.0 * u.mV),
)
cell_named.paint(
    BranchSlice(branch_index=1, prox=0.0, dist=1.0),
    Ion("SodiumFixed", name="na_dend", E=45.0 * u.mV),
)
cell_named.paint(
    BranchSlice(branch_index=0, prox=0.0, dist=1.0),
    Channel("INa_HH1952", ion_name="na_soma", g_max=12.0 * (u.mS / u.cm**2)),
)

print("layout kinds:", [layout.kind for layout in cell_named.layouts])
print("na_soma name:", cell_named.get_ion("na_soma").name)
print("na_soma E shape:", cell_named.get_ion("na_soma").E.shape)

try:
    cell_named.get_ion("na")
except Exception as exc:
    print(type(exc).__name__, exc)


layout kinds: ['ion:SodiumFixed', 'channel:INa_HH1952', 'ion:SodiumFixed']
na_soma name: na_soma
na_soma E shape: (7,)
ValueError Ion selector 'na' is ambiguous; family 'na' has candidates ['na_soma', 'na_dend'].


## 3. Mixed-ion channel (`K + Ca`)

`IKca3_1_Ma2020` depends on potassium and calcium.
Here potassium is the current owner, while calcium only modulates channel gating.

For mixed-ion channels the selector is `ion_names={...}` and it only needs to specify the ambiguous family.

In [5]:
cell_mixed = Cell(build_demo_morphology())
cell_mixed.paint(
    BranchSlice(branch_index=[0, 1, 2], prox=0.0, dist=1.0),
    Ion("PotassiumFixed", name="k_main", E=-88.0 * u.mV),
)
cell_mixed.paint(
    BranchSlice(branch_index=0, prox=0.0, dist=1.0),
    Ion("CalciumFixed", name="ca_hva", Ci=2e-4 * u.mM),
)
cell_mixed.paint(
    BranchSlice(branch_index=1, prox=0.0, dist=1.0),
    Ion("CalciumFixed", name="ca_lva", Ci=5e-4 * u.mM),
)
cell_mixed.paint(
    BranchSlice(branch_index=[0, 1], prox=0.0, dist=1.0),
    Channel("IKca3_1_Ma2020", ion_names={"ca": "ca_hva"}),
)
cell_mixed.place(
    at("soma", 0.5),
    CurrentProbe(mechanism="IKca3_1_Ma2020"),
    CurrentProbe(ion="k_main"),
)
cell_mixed.init_state()
samples = cell_mixed.sample_probes()

print(sorted(samples))
print("owner has channel:", "IKca3_1_Ma2020" in cell_mixed.get_ion("k_main").channels)
print("modulator has channel:", "IKca3_1_Ma2020" in cell_mixed.get_ion("ca_hva").channels)


['soma(0.5)_IKca3_1_Ma2020_current', 'soma(0.5)_k_main_current']
owner has channel: True
modulator has channel: False


## Notes

- `cell.get_ion("family")` only works when that family is unique in the current `Cell`.
- `ion_name` is for single-ion channels.
- `ion_names={...}` is for mixed-ion channels.
- Dynamic ion mechanisms and the Cerebellum calcium-dynamics import path are later steps; this notebook only demonstrates the current routing model.
